In [ ]:
import anndata
import pickle
from scipy.sparse import csc_matrix

In [ ]:
raw_list = []
scvi_list = [] 

scvi = anndata.read_h5ad(snakemake.input["scvi"])

In [ ]:
# normalize to CPM and set 0 counts
# scvi.X = (10000 * scvi.X.T / scvi.X.sum(axis=1)).T
scvi.X[scvi.X < 0.5] = 0
scvi.X = csc_matrix(scvi.X)

In [ ]:
scvi.X.eliminate_zeros()

In [ ]:
import numpy as np
dense_matrix_1 = scvi.X.toarray().flatten()
dense_matrix_2 = scvi.layers["raw"].toarray()

# Stack the dense matrices along a new axis to create a 3D array
combined_array = np.array([dense_matrix_1.flatten(), dense_matrix_2.flatten()])


In [ ]:
combined_array = np.array([dense_matrix_1.flatten(), dense_matrix_2.flatten()])

In [ ]:
filtered = combined_array[:, (combined_array >= 1).any(axis=0)]

In [ ]:
filtered.shape

In [ ]:
combined_array.shape

In [ ]:
random_col_indices = np.random.choice(filtered.shape[1], size=10000, replace=False)
sampled_columns = filtered[:, random_col_indices]


In [ ]:
# plot correlation between .X and .raw



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

data = pd.DataFrame(
    columns=["x", "y"],
    data=np.log(sampled_columns.T+1)
)


# Scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=data, x='x', y='y')
plt.title('Scatter Plot of x vs y')
plt.show()

# Hexbin plot with a density scale
plt.figure(figsize=(10, 6))
plt.hexbin(data['x'], data['y'], gridsize=50, cmap='Blues')
plt.colorbar(label='Density')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Hexbin Plot of x vs y')
plt.show()

# 2D histogram
plt.figure(figsize=(10, 6))
plt.hist2d(data['x'], data['y'], bins=(50, 50), cmap=plt.cm.jet)
plt.colorbar(label='Density')
plt.xlabel('x')
plt.ylabel('y')
plt.title('2D Histogram of x vs y')
plt.show()


In [ ]:
scvi.obs.iloc[0]

In [ ]:
int(np.median([5, 12]))

In [ ]:
if "anno_ordered" in scvi.obs.columns:
    scvi.obs["anno_time"] = scvi.obs["anno_ordered"].apply(lambda s: f"day 01-05 {s[3:]}")  # cut the ordering
else:
    scvi.obs["anno_time"] = scvi.obs["anno_og_time"].apply(lambda s: f"day {s[1:]}" if s != "undefined" else "day 01-42 unknown time")
# adata.var["gene_name"] = adata.var.index  # doesn't work for some reason

In [ ]:
import re

def str_to_day(s):
    vals = re.findall("[0-9]+", s)
    return int(np.mean([int(v) for v in vals]))


scvi.obs["anno_time_cont"] = scvi.obs.anno_time.apply(str_to_day)


In [ ]:
scvi.obs.anno_time_cont.value_counts()

In [ ]:
scvi.obs.anno_time.value_counts()

In [ ]:
scvi.var["gene_name"] = scvi.var.index

In [ ]:
scvi.write_h5ad(snakemake.output.read_count_table)
